In [ ]:
# Cell 0: Install dependencies (run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '../requirements.txt'])

# Azalyst Alpha Research Engine v4.0 — LOCAL GPU (RTX 2050)

**Built by Azalyst Research** | Cross-Sectional Crypto Alpha with XGBoost CUDA

### Local GPU Configuration
- **GPU**: NVIDIA RTX 2050 (4GB GDDR6) — CUDA pinned to device 0
- **CPU**: Intel i5-11260H (6C/12T)  
- **VRAM Guard**: 2M rows max (prevents OOM on 4GB)
- **Dual CUDA API**: Supports both `device='cuda'` (new) and `tree_method='gpu_hist'` (old)

### v4 Architecture
1. **Expanding Window Training** — Y1 → Y1+Y2 → Y1+Y2+Y3
2. **Regime-Aware Feature Selection** — Drop features with negative rolling IC
3. **Risk Integration** — VaR/CVaR + HRP position sizing
4. **Drawdown Kill-Switch** — Halts trading at -15% max DD
5. **SHAP Explainability** — Computed after every training cycle
6. **SQLite Persistence** — All results stored in azalyst.db
7. **Meta-Labeling** — Confidence-weighted sizing (AFML Ch. 3)

**56 Features** → IC Selection → XGBoost CUDA → Purged K-Fold → Expanding Window → Walk Y2+Y3

### Output Directory
`C:/Users/Administrator/Music/azalyst_jupyter_output/`

In [ ]:
# ── Cell 1: Imports & Paths (v4) ──────────────────────────────────────────────
import os, sys, time, json, gc, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sp_stats
warnings.filterwarnings('ignore')

# Add parent directory for v4 module imports
sys.path.insert(0, str(Path('..').resolve()))

import azalyst_v4_engine as v4
from azalyst_factors_v2 import build_features, FEATURE_COLS
from azalyst_risk import RiskManager
from azalyst_db import AzalystDB
from azalyst_v4_engine import (
    detect_cuda_api, make_xgb_params, PurgedTimeSeriesCV,
    build_training_matrix, train_model, train_meta_model,
    load_feature_store, build_feature_store, get_date_splits,
    inspect_feature_store, detect_regime, compute_shap, save_shap_csv,
    compute_feature_ic, select_features_by_ic,
    predict_week, simulate_weekly_trades, compute_drawdown,
    _fix_timestamp,
)

# Override paths for local GPU notebook
v4.DATA_DIR    = str(Path('..', 'data').resolve())
v4.CACHE_DIR   = str(Path('..', 'feature_cache').resolve())
v4.RESULTS_DIR = r'C:/Users/Administrator/Music/azalyst_jupyter_output'

RESULTS_DIR = v4.RESULTS_DIR

for d in [RESULTS_DIR, f'{RESULTS_DIR}/models', f'{RESULTS_DIR}/shap']:
    os.makedirs(d, exist_ok=True)

print(f'DATA_DIR    : {v4.DATA_DIR}')
print(f'CACHE_DIR   : {v4.CACHE_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'Features    : {len(FEATURE_COLS)}')

In [ ]:
# ── Cell 2: GPU Setup (RTX 2050 — Dual CUDA API) ──────────────────────────────
import subprocess, xgboost as xgb

# Pin to RTX 2050 (device 0) — ignore Intel UHD integrated GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"

try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    print(r.stdout.decode()[:300])
except: pass

CUDA_API = detect_cuda_api()
HAS_GPU  = CUDA_API is not None

import psutil
def mem_gb():
    return psutil.virtual_memory().used / 1e9

print(f'XGBoost {xgb.__version__}: CUDA_API={CUDA_API} | HAS_GPU={HAS_GPU}')

In [ ]:
# ── Cell 3: System Diagnostics ───────────────────────────────────────────────
import psutil, platform
print(f'Platform  : {platform.system()} {platform.release()}')
print(f'CPU       : {platform.processor()}')
print(f'RAM       : {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'GPU       : {"RTX 2050 CUDA" if HAS_GPU else "CPU only"}')
print(f'CUDA API  : {CUDA_API or "none"}')
print(f'XGBoost   : {xgb.__version__}')
print(f'VRAM cap  : 2,000,000 rows')

In [ ]:
# ── Cell 4: v4 Configuration ──────────────────────────────────────────────────
from azalyst_v4_engine import (
    MAX_TRAIN_ROWS, RETRAIN_WEEKS, TOP_QUANTILE,
    FEE_RATE, ROUND_TRIP_FEE, HORIZON_BARS,
    MAX_DRAWDOWN_KILL, IC_SELECTION_THRESH, IC_LOOKBACK_WEEKS,
    MIN_FEATURES, VAR_CONFIDENCE, POSITION_RISK_CAP,
)

print(f'[Cell 4] v4 Config loaded.')
print(f'  Features       : {len(FEATURE_COLS)}')
print(f'  GPU mode       : {HAS_GPU}')
print(f'  Retrain every  : {RETRAIN_WEEKS} weeks')
print(f'  Fee round-trip : {ROUND_TRIP_FEE*100:.2f}%')
print(f'  Max DD kill    : {MAX_DRAWDOWN_KILL*100:.0f}%')
print(f'  IC thresh      : {IC_SELECTION_THRESH}')
print(f'  VRAM guard     : {MAX_TRAIN_ROWS:,} rows')
print(f'  Train strategy : Expanding window (Y1 → Y1+Y2 → Y1+Y2+Y3)')
print(f'  Walk strategy  : Y2 + Y3 (2-year OOS)')

In [ ]:
# ── Cell 5: Build / Validate Feature Store ────────────────────────────────────
print('[Cell 5] Checking feature store...')
total, valid, invalid = inspect_feature_store()
print(f'  Total: {total} | Valid: {valid} | Invalid: {invalid}')

if valid < 10:
    print('  Building feature store from raw data...')
    build_feature_store()
    total, valid, invalid = inspect_feature_store()

print(f'  Feature store ready: {valid} valid symbols')
gc.collect()

In [ ]:
# ── Cell 6: Load Feature Store + Date Splits ─────────────────────────────────
print('[Cell 6] Loading feature store...')
t0 = time.time()

symbols = load_feature_store()
if not symbols:
    raise RuntimeError('No valid symbols loaded. Check feature_cache/ directory.')

global_min, global_max, y1_end, y2_end = get_date_splits(symbols)

print(f'\n  Loaded {len(symbols)} symbols in {time.time()-t0:.1f}s')
print(f'  RAM: {mem_gb():.1f} GB')
print(f'\n  v4 Training Strategy:')
print(f'    Init train : Y1 ({global_min.date()} -> {y1_end.date()})')
print(f'    Walk Y2    : {y1_end.date()} -> {y2_end.date()}')
print(f'    Walk Y3    : {y2_end.date()} -> {global_max.date()}')
gc.collect()

In [ ]:
# ── Cell 7: v4 Initial Training on Y1 (Expanding Window Start) ────────────────
print('[Cell 7] Training initial model on Y1 data...')
print(f'  Train window: {global_min.date()} -> {y1_end.date()}')
print(f'  GPU: {HAS_GPU} (CUDA_API={CUDA_API})')

# Initialize SQLite persistence
db = AzalystDB(str(Path(RESULTS_DIR) / 'azalyst.db'))
run_id = db.start_run(config={
    'version': 'v4.0', 'gpu': HAS_GPU, 'cuda_api': CUDA_API,
    'retrain_weeks': RETRAIN_WEEKS, 'max_dd_kill': MAX_DRAWDOWN_KILL,
    'ic_thresh': IC_SELECTION_THRESH, 'features': len(FEATURE_COLS),
})

# Build training matrix (Y1 only — expanding window initial state)
active_features = list(FEATURE_COLS)
X_train, y_train, y_ret = build_training_matrix(symbols, y1_end, active_features)
if X_train is None:
    raise RuntimeError('Failed to build initial training matrix.')

# Train base model
t0 = time.time()
BASE_MODEL, BASE_SCALER, importance, mean_auc, mean_ic, icir = train_model(
    X_train, y_train, y_ret, CUDA_API, active_features, label='base_y1'
)
elapsed = time.time() - t0

# Save base model
os.makedirs(f'{RESULTS_DIR}/models', exist_ok=True)
BASE_MODEL.save_model(f'{RESULTS_DIR}/models/model_v4_base.json')
with open(f'{RESULTS_DIR}/models/scaler_v4_base.pkl', 'wb') as f:
    pickle.dump(BASE_SCALER, f)
importance.to_csv(f'{RESULTS_DIR}/feature_importance_v4_base.csv')

print(f'\n  -- Base Model (Y1) --')
print(f'  AUC (CV)  : {mean_auc:.4f}')
print(f'  IC  (CV)  : {mean_ic:.4f}')
print(f'  ICIR      : {icir:.4f}')
print(f'  Elapsed   : {elapsed:.1f}s')
print(f'  Top 5: {list(importance.head(5).index)}')

# SHAP
print('\n  Computing SHAP values...')
Xs_shap = BASE_SCALER.transform(X_train)
shap_vals = compute_shap(BASE_MODEL, Xs_shap, active_features)
if shap_vals:
    save_shap_csv(shap_vals, f'{RESULTS_DIR}/models/', 'v4_base')

# Meta model
print('\n  Training meta-labeling model...')
META_MODEL, META_SCALER = train_meta_model(
    BASE_MODEL, BASE_SCALER, X_train, y_train, CUDA_API,
    active_features, label='meta_y1'
)
if META_MODEL is not None:
    with open(f'{RESULTS_DIR}/models/meta_v4_base.pkl', 'wb') as f:
        pickle.dump(META_MODEL, f)
    with open(f'{RESULTS_DIR}/models/meta_scaler_v4_base.pkl', 'wb') as f:
        pickle.dump(META_SCALER, f)

with open(f'{RESULTS_DIR}/train_summary_v4.json', 'w') as f:
    json.dump({'mean_auc': round(mean_auc,5), 'mean_ic': round(mean_ic,5),
               'icir': round(icir,5), 'n_rows': int(len(X_train)),
               'n_features': len(active_features), 'use_gpu': HAS_GPU}, f, indent=2)

del X_train, y_train, y_ret, Xs_shap
gc.collect()
print(f'\n[Cell 7] Done. RAM: {mem_gb():.1f} GB')

In [ ]:
# ── Cell 8: v4 Walk-Forward Y2+Y3 (Expanding Window + Regime-Aware IC) ────────
t0_wf = time.time()
print('[Cell 8] Starting v4 walk-forward...')
print(f'  Walk: Y2 ({y1_end.date()}) + Y3 ({y2_end.date()}) -> {global_max.date()}')
print(f'  Retrain: every {RETRAIN_WEEKS} weeks (expanding window)')
print(f'  Kill-switch: {MAX_DRAWDOWN_KILL*100:.0f}% max DD')
print(f'  Meta: {"enabled" if META_MODEL is not None else "disabled"}')
print()

weeks = pd.date_range(start=y1_end, end=global_max, freq='W-MON')
if len(weeks) < 3:
    raise RuntimeError('Not enough walk-forward weeks.')

current_model  = BASE_MODEL
current_scaler = BASE_SCALER
current_meta   = META_MODEL
current_meta_scaler = META_SCALER
risk_mgr       = RiskManager()
retrain_count  = 0

all_trades_list      = []
weekly_summary_list  = []
weekly_returns_hist  = []
prev_longs, prev_shorts = set(), set()
feature_ic_history = {}
kill_switch_hit = False

for week_num, (ws, we) in enumerate(zip(weeks[:-1], weeks[1:]), 1):
    # Kill-switch check
    current_dd = compute_drawdown(weekly_returns_hist)
    if current_dd < MAX_DRAWDOWN_KILL:
        print(f'\n  *** KILL SWITCH *** Week {week_num}: DD={current_dd*100:.1f}%')
        print('  Halting all trading. Pausing for 4 weeks...')
        kill_switch_hit = True
        weekly_returns_hist.extend([0.0] * min(4, len(weeks) - week_num - 1))
        for skip in range(4):
            weekly_summary_list.append({
                'week': week_num + skip, 'week_start': str(ws.date()),
                'week_end': str(we.date()), 'n_symbols': 0, 'n_trades': 0,
                'week_return_pct': 0.0, 'ic': 0.0, 'turnover_pct': 0.0,
                'regime': 'KILL_SWITCH', 'retrained': False,
            })
        kill_switch_hit = False
        continue

    # Regime detection
    regime = detect_regime(symbols, we)

    # Feature IC tracking (every 2 weeks)
    if week_num > 1 and week_num % 2 == 0:
        fic = compute_feature_ic(symbols, ws, we, active_features)
        for feat, ic_val in fic.items():
            if feat not in feature_ic_history:
                feature_ic_history[feat] = []
            feature_ic_history[feat].append(ic_val)

    # Quarterly retrain with expanding window
    did_retrain = False
    if week_num % RETRAIN_WEEKS == 0:
        print(f'  Week {week_num:3d}: EXPANDING RETRAIN (data up to {we.date()})...')
        try:
            # Regime-aware feature selection
            new_features = select_features_by_ic(feature_ic_history, FEATURE_COLS)
            n_dropped = len(active_features) - len(new_features)
            if n_dropped > 0:
                print(f'    IC filter: {len(active_features)} -> {len(new_features)} features')
            active_features = new_features

            # Expanding window: all data up to current week
            X_rt, y_rt, yret_rt = build_training_matrix(symbols, we, active_features)
            if X_rt is not None and len(X_rt) > 200:
                t0 = time.time()
                m_new, s_new, imp_new, auc_n, ic_n, icir_n = train_model(
                    X_rt, y_rt, yret_rt, CUDA_API, active_features,
                    label=f'v4_w{week_num:03d}'
                )
                current_model, current_scaler = m_new, s_new
                retrain_count += 1
                m_new.save_model(f'{RESULTS_DIR}/models/model_v4_week{week_num:03d}.json')
                imp_new.to_csv(f'{RESULTS_DIR}/feature_importance_v4_week{week_num:03d}.csv')
                print(f'    AUC={auc_n:.4f}  IC={ic_n:.4f}  ICIR={icir_n:.4f}  ({time.time()-t0:.1f}s)')

                # SHAP on retrained model
                Xs_rt = s_new.transform(X_rt)
                shap_v = compute_shap(m_new, Xs_rt, active_features)
                if shap_v:
                    save_shap_csv(shap_v, f'{RESULTS_DIR}/models/', f'v4_week{week_num:03d}')

                # Retrain meta
                meta_new, meta_s_new = train_meta_model(
                    current_model, current_scaler, X_rt, y_rt, CUDA_API,
                    active_features, label=f'meta_w{week_num:03d}'
                )
                if meta_new is not None:
                    current_meta, current_meta_scaler = meta_new, meta_s_new

                del X_rt, y_rt, yret_rt, Xs_rt
                gc.collect()
                did_retrain = True
        except Exception as e:
            print(f'    Retrain failed: {e}')

    # Predict this week
    predictions, actual_rets, meta_confs = predict_week(
        current_model, current_scaler, symbols, ws, we,
        active_features, meta_model=current_meta,
        meta_scaler=current_meta_scaler
    )

    if len(predictions) < 5:
        weekly_returns_hist.append(0.0)
        weekly_summary_list.append({
            'week': week_num, 'week_start': str(ws.date()), 'week_end': str(we.date()),
            'n_symbols': len(predictions), 'n_trades': 0, 'week_return_pct': 0.0,
            'ic': 0.0, 'turnover_pct': 0.0, 'regime': regime, 'retrained': did_retrain,
        })
        continue

    # Trade simulation with risk integration
    trades, week_ret, cur_longs, cur_shorts = simulate_weekly_trades(
        predictions, actual_rets, prev_longs, prev_shorts,
        meta_confs, risk_manager=risk_mgr,
        weekly_returns_hist=weekly_returns_hist
    )
    weekly_returns_hist.append(week_ret)

    # IC
    common = [s for s in predictions if s in actual_rets]
    if len(common) > 10:
        pred_arr = np.array([predictions[s] for s in common])
        ret_arr  = np.array([actual_rets[s] for s in common])
        week_ic  = float(sp_stats.spearmanr(pred_arr, ret_arr)[0])
    else:
        week_ic = 0.0

    # Turnover
    n_cur = len(cur_longs) + len(cur_shorts)
    n_new = len(cur_longs - prev_longs) + len(cur_shorts - prev_shorts)
    turnover = round(n_new / max(n_cur, 1) * 100, 1)
    prev_longs, prev_shorts = cur_longs, cur_shorts

    # Cumulative stats
    cum_ret = float(np.prod([1 + r for r in weekly_returns_hist]) - 1)
    max_dd  = compute_drawdown(weekly_returns_hist)

    for t in trades:
        t['week'] = week_num
        t['week_start'] = str(ws.date())
    all_trades_list.extend(trades)

    zone = 'Y2' if we <= y2_end else 'Y3'
    ann_proj = ((1 + week_ret) ** 52 - 1) * 100

    weekly_summary_list.append({
        'week': week_num, 'week_start': str(ws.date()), 'week_end': str(we.date()),
        'n_symbols': len(predictions), 'n_trades': len(trades),
        'week_return_pct': round(week_ret * 100, 4),
        'annualised_pct': round(ann_proj, 2), 'ic': round(week_ic, 5),
        'turnover_pct': turnover, 'cum_return_pct': round(cum_ret * 100, 4),
        'max_drawdown_pct': round(max_dd * 100, 4),
        'regime': regime, 'retrained': did_retrain,
    })

    # Store in DB
    db.insert_trades(run_id, trades)
    db.insert_weekly_metric(run_id, weekly_summary_list[-1])

    # Progress
    if week_num % 4 == 0 or week_num <= 2:
        rolling = np.mean(weekly_returns_hist[-4:]) * 100 if weekly_returns_hist else 0
        print(f'  Week {week_num:3d} [{zone}] | ret={week_ret*100:+.2f}%  IC={week_ic:+.4f}  '
              f'cum={cum_ret*100:+.1f}%  DD={max_dd*100:.1f}%  n={len(trades)}  {regime}')

    if week_num % 13 == 0:
        print(f'  [Resource] RAM={mem_gb():.1f}GB  Elapsed={( time.time()-t0_wf)/60:.0f}min')
        gc.collect()

db.finish_run(run_id, 'killed' if kill_switch_hit else 'completed')
print(f'\n[Cell 8] v4 Walk-forward complete.')
print(f'  Weeks: {len(weekly_summary_list)} | Trades: {len(all_trades_list)} | Retrains: {retrain_count}')
if kill_switch_hit:
    print(f'  Kill-switch was triggered during run.')
gc.collect()

In [ ]:
# ── Cell 9: Save Results & Performance Metrics (v4) ───────────────────────────
print('[Cell 9] Saving results...')

if not weekly_summary_list:
    print('[WARN] No weekly results. Did Cell 8 run successfully?')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)

    weekly_df.to_csv(f'{RESULTS_DIR}/weekly_summary_v4.csv', index=False)
    trades_df.to_csv(f'{RESULTS_DIR}/all_trades_v4.csv', index=False)

    rets  = weekly_df['week_return_pct'].dropna() / 100
    pnls  = trades_df['pnl_percent'].dropna() / 100 if len(trades_df) > 0 else pd.Series(dtype=float)
    ic_s  = weekly_df['ic'].dropna()

    cum     = (1 + rets).cumprod()
    n_wks   = max(len(rets), 1)
    t_ret   = float(cum.iloc[-1] - 1) if len(cum) > 0 else 0.0
    ann_ret = (1 + t_ret) ** (52 / n_wks) - 1
    sharpe  = float(rets.mean() / (rets.std() + 1e-9) * np.sqrt(52))
    max_dd  = float(((cum - cum.cummax()) / (1 + cum.cummax())).min()) if len(cum) > 0 else 0.0
    win_rt  = float((pnls > 0).mean() * 100) if len(pnls) > 0 else 0.0
    ic_mean = float(ic_s.mean()) if len(ic_s) > 0 else 0.0
    ic_std  = float(ic_s.std())  if len(ic_s) > 0 else 0.0
    icir_v  = float(ic_mean / (ic_std + 1e-8))
    ic_pos  = float((ic_s > 0).mean() * 100) if len(ic_s) > 0 else 0.0
    avg_to  = float(weekly_df['turnover_pct'].mean()) if 'turnover_pct' in weekly_df.columns else 0.0

    # VaR / CVaR
    ret_series = pd.Series([r/100 for r in weekly_df['week_return_pct'].dropna()])
    var_95  = float(risk_mgr.calculate_var(ret_series, 0.95)) if len(ret_series) > 4 else 0.0
    cvar_95 = float(risk_mgr.calculate_cvar(ret_series, 0.95)) if len(ret_series) > 4 else 0.0

    # Y2 vs Y3 analysis
    y2_mask = weekly_df['week_end'].apply(lambda x: pd.Timestamp(x, tz='UTC') <= y2_end if x else False)
    y3_mask = ~y2_mask
    y2_ret = float(weekly_df.loc[y2_mask, 'week_return_pct'].sum()) if y2_mask.any() else 0.0
    y3_ret = float(weekly_df.loc[y3_mask, 'week_return_pct'].sum()) if y3_mask.any() else 0.0
    y2_ic  = float(weekly_df.loc[y2_mask, 'ic'].mean()) if y2_mask.any() else 0.0
    y3_ic  = float(weekly_df.loc[y3_mask, 'ic'].mean()) if y3_mask.any() else 0.0

    performance = {
        'label': 'v4_WalkForward_Y2Y3',
        'total_weeks': n_wks, 'total_trades': len(trades_df), 'retrains': retrain_count,
        'total_return_pct': round(t_ret * 100, 2),
        'annualised_pct': round(ann_ret * 100, 2),
        'sharpe': round(sharpe, 4),
        'max_drawdown_pct': round(max_dd * 100, 2),
        'win_rate_pct': round(win_rt, 2),
        'ic_mean': round(ic_mean, 5),
        'icir': round(icir_v, 4),
        'ic_positive_pct': round(ic_pos, 1),
        'avg_turnover_pct': round(avg_to, 1),
        'var_95': round(var_95 * 100, 4),
        'cvar_95': round(cvar_95 * 100, 4),
        'y2_return_pct': round(y2_ret, 2),
        'y3_return_pct': round(y3_ret, 2),
        'y2_ic_mean': round(y2_ic, 5),
        'y3_ic_mean': round(y3_ic, 5),
        'meta_labeling': META_MODEL is not None,
        'kill_switch': MAX_DRAWDOWN_KILL,
        'use_gpu': HAS_GPU,
    }
    with open(f'{RESULTS_DIR}/performance_v4.json', 'w') as f:
        json.dump(performance, f, indent=2)

    # Persist to SQLite
    db.insert_performance_summary(run_id, performance)

    print(f'\n  ── v4 Walk-Forward Performance (Y2+Y3 OOS) ──')
    print(f'  Total Return   : {performance["total_return_pct"]:>8.2f}%')
    print(f'  Annualised     : {performance["annualised_pct"]:>8.2f}%')
    print(f'  Sharpe         : {performance["sharpe"]:>8.4f}')
    print(f'  Max Drawdown   : {performance["max_drawdown_pct"]:>8.2f}%')
    print(f'  Win Rate       : {performance["win_rate_pct"]:>8.2f}%')
    print(f'  IC Mean        : {performance["ic_mean"]:>8.5f}')
    print(f'  ICIR           : {performance["icir"]:>8.4f}')
    print(f'  VaR (95%)      : {performance["var_95"]:>8.4f}%')
    print(f'  CVaR (95%)     : {performance["cvar_95"]:>8.4f}%')
    print(f'  Y2 Return      : {performance["y2_return_pct"]:>8.2f}%  (IC={y2_ic:.4f})')
    print(f'  Y3 Return      : {performance["y3_return_pct"]:>8.2f}%  (IC={y3_ic:.4f})')
    print(f'  Avg Turnover   : {performance["avg_turnover_pct"]:>8.1f}%')
    print(f'  Retrains       : {retrain_count}')
    print(f'  Meta-labeling  : {performance["meta_labeling"]}')

In [ ]:
# ── Cell 10: Charts + ZIP (v4) ────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import zipfile

print('[Cell 10] Generating v4 performance charts...')

if not weekly_summary_list:
    print('[WARN] No results to chart.')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)

    fig = plt.figure(figsize=(16, 11))
    fig.suptitle('Azalyst v4.0 — Walk-Forward Y2+Y3 (Expanding Window + IC Selection + Kill-Switch)',
                 fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    # Panel 1: Cumulative return with Y2/Y3 zones
    ax1 = fig.add_subplot(gs[0, 0])
    rets = weekly_df['week_return_pct'].fillna(0) / 100
    cum  = ((1 + rets).cumprod() - 1) * 100
    ax1.plot(weekly_df['week'], cum, color='#1f77b4', linewidth=2)
    ax1.fill_between(weekly_df['week'], cum, alpha=0.12, color='#1f77b4')
    # Mark Y2/Y3 boundary
    y2_weeks = len(weekly_df[weekly_df['week_end'].apply(
        lambda x: pd.Timestamp(x, tz='UTC') <= y2_end if x else False)])
    if y2_weeks > 0 and y2_weeks < len(weekly_df):
        ax1.axvline(y2_weeks, color='red', linewidth=1.5, linestyle='--', alpha=0.6, label='Y2→Y3')
        ax1.legend(fontsize=9)
    ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax1.set_title('Cumulative Return (%)', fontweight='bold')
    ax1.set_xlabel('Week #'); ax1.set_ylabel('%'); ax1.grid(True, alpha=0.25)

    # Panel 2: Weekly return distribution
    ax2 = fig.add_subplot(gs[0, 1])
    wr = weekly_df['week_return_pct'].dropna()
    ax2.hist(wr, bins=min(30, max(10, len(wr)//3)), color='#ff7f0e', alpha=0.72,
             edgecolor='black', linewidth=0.4)
    if len(wr) > 2:
        ax2.axvline(wr.mean(),   color='red',   linewidth=1.8, linestyle='--', label=f'Mean {wr.mean():.2f}%')
        ax2.axvline(wr.median(), color='green', linewidth=1.2, linestyle=':',  label=f'Median {wr.median():.2f}%')
        ax2.legend(fontsize=9)
    ax2.set_title('Weekly Return Distribution', fontweight='bold')
    ax2.set_xlabel('Weekly Return (%)'); ax2.set_ylabel('Count'); ax2.grid(True, alpha=0.25)

    # Panel 3: IC series
    ax3 = fig.add_subplot(gs[1, 0])
    ic_vals = weekly_df['ic'].fillna(0)
    ax3.bar(weekly_df['week'], ic_vals,
            color=['#2ca02c' if v > 0 else '#d62728' for v in ic_vals], alpha=0.75, width=0.8)
    if len(ic_vals) > 2:
        ax3.axhline(ic_vals.mean(), color='navy', linewidth=1.5, linestyle='--',
                    label=f'Mean IC {ic_vals.mean():.4f}')
        ax3.legend(fontsize=9)
    ax3.axhline(0, color='black', linewidth=0.6)
    ax3.set_title('Weekly IC (Information Coefficient)', fontweight='bold')
    ax3.set_xlabel('Week #'); ax3.set_ylabel('Spearman IC'); ax3.grid(True, alpha=0.25)

    # Panel 4: Trade P&L
    ax4 = fig.add_subplot(gs[1, 1])
    if len(trades_df) > 0 and 'pnl_percent' in trades_df.columns:
        pnl = trades_df['pnl_percent'].dropna()
        ax4.hist(pnl, bins=min(40, max(10, len(pnl)//20)), color='#9467bd', alpha=0.72,
                 edgecolor='black', linewidth=0.3)
        ax4.axvline(pnl.mean(), color='red', linewidth=1.8, linestyle='--',
                    label=f'Mean {pnl.mean():.3f}%')
        ax4.axvline(0, color='black', linewidth=0.8)
        ax4.legend(fontsize=9)
        ax4.set_title(f'Trade P&L Distribution (n={len(pnl):,})', fontweight='bold')
        ax4.set_xlabel('P&L per Trade (%)'); ax4.set_ylabel('Count'); ax4.grid(True, alpha=0.25)

    chart_path = f'{RESULTS_DIR}/performance_v4.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Chart saved -> {chart_path}')

    # ZIP results
    zip_path = f'{RESULTS_DIR}/azalyst_v4_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in Path(RESULTS_DIR).rglob('*'):
            if fp.is_file() and fp.suffix in ('.csv', '.json', '.png', '.pkl', '.db'):
                zf.write(fp, fp.relative_to(Path(RESULTS_DIR).parent))
    zip_mb = Path(zip_path).stat().st_size / 1e6
    print(f'  Results ZIP -> {zip_path}  ({zip_mb:.1f} MB)')

    print(f'''
=========================================================
  AZALYST v4.0 — RUN COMPLETE
  Built by Azalyst Research
=========================================================
  Total Return   : {performance["total_return_pct"]:.2f}%
  Annualised     : {performance["annualised_pct"]:.2f}%
  Sharpe         : {performance["sharpe"]:.4f}
  Max Drawdown   : {performance["max_drawdown_pct"]:.2f}%
  IC Mean        : {performance["ic_mean"]:.5f}
  ICIR           : {performance["icir"]:.4f}
  VaR (95%)      : {performance["var_95"]:.4f}%
  CVaR (95%)     : {performance["cvar_95"]:.4f}%
  Y2 Return      : {performance["y2_return_pct"]:.2f}%
  Y3 Return      : {performance["y3_return_pct"]:.2f}%
  Kill-Switch    : {MAX_DRAWDOWN_KILL*100:.0f}%
  Meta-labeling  : {performance["meta_labeling"]}
  GPU used       : {HAS_GPU}
=========================================================
  v4 Architecture:
    1. Expanding Window Training
    2. Regime-Aware IC Feature Selection
    3. Risk Integration (VaR/CVaR + HRP)
    4. Drawdown Kill-Switch
    5. SHAP Explainability
    6. SQLite Persistence
=========================================================
  Download azalyst_v4_results.zip from Output tab.
''')